# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a practical guide for loading and exploring the Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

# Access metadata fields directly from the .metadata object
print(f"Dataset Name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")


## 2. Data Overview
Review available record sets, their IDs, and field structures.

Entities in Croissant are referenced via their `@id` field. Let's list record sets and a sample of their fields via `@id` for later referencing.


In [ ]:
# List all record sets and their key attributes by @id

record_sets = dataset.metadata.record_sets
print(f"Record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set Name: {getattr(rs, 'name', None)}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', None)}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            label = getattr(field, 'name', None) or getattr(field, 'title', None)
            print(f"    - {label}: @id={field.id}, Type={getattr(field, 'data_type', None)}")
    print()


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the `@id` field for the record set and the field(s) you are interested in.


In [ ]:
# Prepare a list of record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from Record Set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  Number of records: {len(df)}\n")
    # Display the first few rows for sample
    display(df.head())


## 4. Exploratory Data Analysis (EDA)
We'll process a numeric column from one of the record sets. Please replace the placeholders below with actual field `@id`s and record set `@id` from the previous cell's results.
- Filtering records based on a numeric field (e.g., 'coverage_percent' or 'abundance')
- Normalizing that field
- Grouping by a categorical field (e.g., 'sample_id', if available)


In [ ]:
# Choose one main record set to analyze (update variable as needed)
main_record_set_id = record_set_ids[0]  # <-- Replace with the record set you wish to analyze
df = dataframes[main_record_set_id]
print(f"Analyzing data from Record Set: {main_record_set_id}")

# List available columns for choosing numeric/categorical fields
print("Available columns:", df.columns.tolist())

# Choose a numeric field for filtering and normalization (update variable as needed)
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    print("No numeric field detected, please update the field selection.")
else:
    print(f"Using numeric field for analysis: {numeric_field_id}")
    threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
    filtered_df = df[df[numeric_field_id] > threshold]

    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Try grouping by the first non-numeric field (update as appropriate)
    group_field_id = None
    for col in df.columns:
        if (not pd.api.types.is_numeric_dtype(df[col])):
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id}, aggregated mean of {numeric_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable group field identified.")


## 5. Visualization
Visualize the distribution of the chosen numeric field and, if applicable, how it varies across groupings.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if 'grouped_df' in locals() and group_field_id is not None:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use `mlcroissant` to:
- Load and inspect a scientific dataset using Croissant schema
- Explore the available record sets and their fields by `@id`
- Extract data from a selected record set and conduct basic EDA (filtering, normalization, grouping)
- Visualize field distributions and relationships

**Note:**
- For production and detailed analysis, update the chosen `record_set_id` and field `@id`s based on the dataset schema and your research interest.
- Always reference fields, record sets, and columns by their Croissant `@id`s for consistency and reproducibility.
- Explore `dataset.metadata` further for comprehensive metadata and documentation on record sets, fields, data provenance, and licensing!
